In [1]:
# ============================================
# LEAGUE OF LEGENDS - ANÁLISIS DE PARTIDAS
# ============================================

import requests
import pandas as pd
from datetime import datetime
from getpass import getpass

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [10]:
# ============================================
# CONFIGURACIÓN
# ============================================

API_KEY = getpass("🔑 Pega aquí tu Riot API Key: ")

GAME_NAME = "La Denko Sekka"
TAG_LINE = "LAN"

# Región de Riot para obtener las partidas
REGION = "americas"

# 420 = Clasificatoria Solo/Dúo
QUEUE_ID = 420

print("✅ Configuración lista")

🔑 Pega aquí tu Riot API Key: ··········
✅ Configuración lista


In [12]:
# PRUEBA DE LA API DE RIOT

import requests

headers = {
    "X-Riot-Token": API_KEY
}

url = (
    "https://americas.api.riotgames.com/"
    "riot/account/v1/accounts/by-riot-id/"
    "La%20Denko%20Sekka/LAN"
)

respuesta = requests.get(url, headers=headers)

print("Código:", respuesta.status_code)
print("Respuesta:", respuesta.text)

Código: 200
Respuesta: {"puuid":"aJfiqnINgIeqIYHayu13NMUj2OMWA-ZwH_JCwucKO8aVSPaZDpKXY44bmEVKJndbe-BqHz5nx8qD3A","gameName":"La Denko Sekka","tagLine":"LAN"}


In [14]:
# ============================================
# OBTENER PUUID
# ============================================

headers = {
    "X-Riot-Token": API_KEY
}

url = (
    f"https://{REGION}.api.riotgames.com/"
    f"riot/account/v1/accounts/by-riot-id/"
    f"{GAME_NAME}/{TAG_LINE}"
)

respuesta = requests.get(url, headers=headers)

if respuesta.status_code != 200:
    print("❌ Error al encontrar la cuenta")
    print("Código:", respuesta.status_code)
    print(respuesta.text)
else:

    cuenta = respuesta.json()

    puuid = cuenta["puuid"]

    print("✅ Cuenta encontrada")
    print("Nombre:", cuenta["gameName"])
    print("Tag:", cuenta["tagLine"])
    print("PUUID obtenido correctamente")

✅ Cuenta encontrada
Nombre: La Denko Sekka
Tag: LAN
PUUID obtenido correctamente


In [15]:
# ============================================
# OBTENER IDs DE PARTIDAS
# ============================================

url = (
    f"https://{REGION}.api.riotgames.com/"
    f"lol/match/v5/matches/by-puuid/"
    f"{puuid}/ids"
)

parametros = {
    "start": 0,
    "count": 100
}

respuesta = requests.get(
    url,
    headers=headers,
    params=parametros
)

if respuesta.status_code != 200:

    print("❌ Error obteniendo las partidas")
    print(respuesta.status_code)
    print(respuesta.text)

else:

    match_ids = respuesta.json()

    print("✅ Partidas encontradas:", len(match_ids))
    print()

    print("Primeras partidas:")

    for match in match_ids[:10]:
        print(match)

✅ Partidas encontradas: 100

Primeras partidas:
LA1_1746975761
LA1_1746972235
LA1_1746930688
LA1_1746915437
LA1_1746712100
LA1_1746702954
LA1_1746662733
LA1_1746462234
LA1_1744533095
LA1_1744432837


In [16]:
# ============================================
# EXTRAER DATOS DE LAS PARTIDAS
# ============================================

partidas = []

print("🔎 Analizando partidas...")
print()

for numero, match_id in enumerate(match_ids):

    print(f"Partida {numero + 1}/{len(match_ids)}", end="\r")

    url = (
        f"https://{REGION}.api.riotgames.com/"
        f"lol/match/v5/matches/{match_id}"
    )

    respuesta = requests.get(
        url,
        headers=headers
    )

    if respuesta.status_code != 200:
        continue

    partida = respuesta.json()

    info = partida["info"]

    # -----------------------------------------
    # SOLO CLASIFICATORIA SOLO/DÚO
    # -----------------------------------------

    if info["queueId"] != QUEUE_ID:
        continue

    # -----------------------------------------
    # ENCONTRAR TU PARTICIPACIÓN
    # -----------------------------------------

    jugador = None

    for participante in info["participants"]:

        if participante["puuid"] == puuid:

            jugador = participante
            break

    if jugador is None:
        continue

    # -----------------------------------------
    # DURACIÓN
    # -----------------------------------------

    duracion = info["gameDuration"] / 60

    # -----------------------------------------
    # KDA
    # -----------------------------------------

    kills = jugador["kills"]
    deaths = jugador["deaths"]
    assists = jugador["assists"]

    if deaths == 0:

        kda = kills + assists

    else:

        kda = (kills + assists) / deaths

    # -----------------------------------------
    # CS
    # -----------------------------------------

    cs = (
        jugador.get("totalMinionsKilled", 0)
        +
        jugador.get("neutralMinionsKilled", 0)
    )

    cs_min = cs / duracion

    # -----------------------------------------
    # ORO
    # -----------------------------------------

    oro = jugador.get(
        "goldEarned",
        0
    )

    oro_min = oro / duracion

    # -----------------------------------------
    # DAÑO
    # -----------------------------------------

    dano = jugador.get(
        "totalDamageDealtToChampions",
        0
    )

    dano_min = dano / duracion

    dano_recibido = jugador.get(
        "totalDamageTaken",
        0
    )

    # -----------------------------------------
    # KILL PARTICIPATION
    # -----------------------------------------

    challenges = jugador.get(
        "challenges",
        {}
    )

    kill_participation = challenges.get(
        "killParticipation",
        None
    )

    if kill_participation is not None:
        kill_participation *= 100

    # -----------------------------------------
    # FECHA
    # -----------------------------------------

    fecha = datetime.fromtimestamp(
        info["gameCreation"] / 1000
    )

    # -----------------------------------------
    # GUARDAR PARTIDA
    # -----------------------------------------

    datos = {

        "match_id": match_id,

        "fecha_hora": fecha,

        "resultado":
            "Victoria"
            if jugador["win"]
            else "Derrota",

        "campeon":
            jugador["championName"],

        "rol":
            jugador.get(
                "teamPosition",
                "UNKNOWN"
            ),

        "kills":
            kills,

        "muertes":
            deaths,

        "asistencias":
            assists,

        "kda":
            round(kda, 2),

        "cs":
            cs,

        "cs_min":
            round(cs_min, 2),

        "oro":
            oro,

        "oro_min":
            round(oro_min, 2),

        "dano_campeones":
            dano,

        "dano_min":
            round(dano_min, 2),

        "dano_recibido":
            dano_recibido,

        "vision_score":
            jugador.get(
                "visionScore",
                0
            ),

        "wards_colocados":
            jugador.get(
                "wardsPlaced",
                0
            ),

        "wards_destruidos":
            jugador.get(
                "wardsKilled",
                0
            ),

        "control_wards":
            jugador.get(
                "controlWardsPlaced",
                0
            ),

        "kill_participation":
            round(kill_participation, 2)
            if kill_participation is not None
            else None,

        "duracion_min":
            round(duracion, 2)
    }

    partidas.append(datos)


# ============================================
# CREAR TABLA
# ============================================

df = pd.DataFrame(partidas)

# Ordenar por fecha
df = df.sort_values(
    "fecha_hora",
    ascending=False
)

print()
print()
print("====================================")
print("✅ ANÁLISIS TERMINADO")
print("====================================")

print()

print(
    "Partidas clasificatorias encontradas:",
    len(df)
)

🔎 Analizando partidas...

Partida 100/100

✅ ANÁLISIS TERMINADO

Partidas clasificatorias encontradas: 52


In [17]:
# Mostrar las primeras partidas

df.head(70)

,match_id,fecha_hora,resultado,campeon,rol,kills,muertes,asistencias,kda,cs,...,oro_min,dano_campeones,dano_min,dano_recibido,vision_score,wards_colocados,wards_destruidos,control_wards,kill_participation,duracion_min
0,LA1_1746975761,2026-09-07 06:24:27.293,Derrota,Maokai,UTILITY,0,5,3,0.60,26,...,254.92,7144,461.40,10990,24,12,1,0,75.00,15.48
1,LA1_1746972235,2026-09-07 05:48:52.931,Victoria,Maokai,UTILITY,3,6,12,2.50,41,...,318.72,12545,434.08,19060,73,23,10,0,34.09,28.90
2,LA1_1746930688,2026-09-07 03:41:18.391,Derrota,Seraphine,UTILITY,3,9,10,1.44,76,...,323.47,17571,618.33,23719,64,26,5,0,50.00,28.42
3,LA1_1746915437,2026-09-07 03:00:52.799,Derrota,Pantheon,UTILITY,2,12,13,1.25,66,...,307.51,17124,533.74,28464,76,28,6,0,65.22,32.08
4,LA1_1746712100,2026-09-06 07:36:00.243,Victoria,Pantheon,UTILITY,7,5,8,3.00,51,...,414.13,12910,541.68,15274,57,26,4,0,41.67,23.83
5,LA1_1746702954,2026-09-06 07:06:49.691,Derrota,Galio,UTILITY,1,11,7,0.73,30,...,278.39,8115,341.21,20089,46,22,6,0,61.54,23.78
6,LA1_1746662733,2026-09-06 04:40:27.005,Derrota,Malzahar,MIDDLE,11,6,16,4.50,404,...,446.76,44643,894.05,31877,55,25,10,0,60.00,49.93
7,LA1_1746462234,2026-09-05 10:44:51.343,Victoria,Galio,UTILITY,2,5,9,2.20,48,...,306.14,9706,318.58,17037,92,42,8,0,40.74,30.47
8,LA1_1744533095,2026-08-28 16:49:52.853,Victoria,Galio,UTILITY,1,3,12,4.33,40,...,341.05,6906,290.17,11207,67,21,8,0,36.11,23.80
9,LA1_1744432837,2026-08-28 03:25:24.558,Victoria,Nami,UTILITY,3,0,10,13.00,27,...,339.11,6072,319.58,5243,30,15,4,0,65.00,19.00
